# Student Performance Analysis and Prediction

**AICTE | IBM SkillsBuild Data Analytics with AI Academic Internship 2026**

**Student:** Rajneesh Verma  
**Branch:** B.Tech - Computer Science and Engineering  
**Institution:** Uttar Pradesh Textile Technology Institute, Kanpur

## Project Overview
This project demonstrates an end-to-end data analytics and machine-learning workflow for analyzing student academic performance and predicting final scores. The notebook uses a reproducible synthetic dataset so that no private student records are exposed.


## Objectives
1. Inspect and clean student-performance data.
2. Perform exploratory data analysis and visualization.
3. Analyze relationships among study time, attendance, previous performance and final score.
4. Train a machine-learning regression model.
5. Evaluate the model using MAE, RMSE and R².
6. Identify important predictive features and document limitations.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)
print("Libraries imported successfully.")


## 1. Dataset
For reproducibility, this notebook creates a synthetic educational dataset. A public UCI Student Performance Dataset is referenced for future extension:

https://archive.ics.uci.edu/dataset/320/student+performance


In [ ]:
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'study_hours': np.round(np.clip(np.random.normal(4.5, 1.8, n), 0.5, 10), 1),
    'attendance_pct': np.round(np.clip(np.random.normal(82, 10, n), 50, 100), 1),
    'previous_score': np.round(np.clip(np.random.normal(68, 12, n), 35, 95), 1),
    'assignments_completed_pct': np.round(np.clip(np.random.normal(78, 14, n), 30, 100), 1),
    'sleep_hours': np.round(np.clip(np.random.normal(7, 1.2, n), 4, 10), 1),
    'internet_access': np.random.choice(['Yes', 'No'], size=n, p=[0.9, 0.1]),
    'extracurricular': np.random.choice(['Yes', 'No'], size=n, p=[0.35, 0.65])
})

noise = np.random.normal(0, 5, n)
df['final_score'] = np.round(np.clip(
    0.32 * df['previous_score'] +
    1.8 * df['study_hours'] +
    0.18 * df['attendance_pct'] +
    0.10 * df['assignments_completed_pct'] +
    1.0 * df['sleep_hours'] +
    2 * (df['internet_access'] == 'Yes') +
    noise, 0, 100), 1)

df.head()


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nDescriptive statistics:")
display(df.describe(include='all').T)


## 2. Data Quality Check


In [ ]:
print("Missing values:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())


## 3. Exploratory Data Analysis
The following charts examine score distribution and relationships between final score and selected predictors.


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['final_score'], bins=20, kde=True)
plt.title('Distribution of Final Scores')
plt.xlabel('Final Score')
plt.ylabel('Number of Students')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='study_hours', y='final_score')
plt.title('Study Hours vs Final Score')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='attendance_pct', y='final_score')
plt.title('Attendance vs Final Score')
plt.show()


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues')
plt.title('Correlation Matrix')
plt.show()


## 4. Feature Preparation
Categorical variables are converted into numeric indicators. The target variable is `final_score`.


In [ ]:
model_df = pd.get_dummies(
    df,
    columns=['internet_access', 'extracurricular'],
    drop_first=True
)

X = model_df.drop(columns='final_score')
y = model_df['final_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


## 5. Machine Learning Model


In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    max_depth=8,
    min_samples_leaf=2
)

model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.3f}")


In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(importance.to_frame('importance'))

plt.figure(figsize=(9, 5))
importance.head(8).sort_values().plot(kind='barh')
plt.title('Top Feature Importances')
plt.xlabel('Importance')
plt.show()


In [ ]:
comparison = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': np.round(predictions, 1)
})
display(comparison.head(15))

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=predictions)
plt.xlabel('Actual Score')
plt.ylabel('Predicted Score')
plt.title('Actual vs Predicted Scores')
plt.show()


## 6. Key Findings
- The analysis examines how study time, attendance, previous performance, assignment completion and other variables relate to final performance.
- The Random Forest model provides a nonlinear baseline for predicting final score.
- Feature importance identifies which available variables contributed most to model predictions.
- Results from this synthetic dataset are educational and should not be used for high-stakes decisions.

## 7. Conclusion
Data analytics can transform student-performance records into useful patterns for academic planning. Combining exploratory analysis with machine learning can support identification of factors associated with academic outcomes. In a real deployment, privacy, fairness, representative data, validation and human oversight would be essential.

## 8. Future Scope
1. Use a larger documented public dataset.
2. Compare multiple machine-learning algorithms.
3. Apply cross-validation and hyperparameter tuning.
4. Build an interactive dashboard using Streamlit or Power BI.
5. Add explainable-AI techniques such as SHAP.
6. Apply privacy and fairness checks before real-world use.

## 9. References
- UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/320/student+performance
- Scikit-learn: https://scikit-learn.org/stable/
- Pandas: https://pandas.pydata.org/docs/
- Matplotlib: https://matplotlib.org/stable/
- Seaborn: https://seaborn.pydata.org/
